# Descarga de D-Fire y verificación de splits

Este notebook:

1. Descarga el dataset `sayedgamal99/smoke-fire-detection-yolo` (D-Fire en formato YOLO) desde Kaggle y lo deja en `data/`.
2. Explora la estructura de carpetas descargada.
3. Cuenta imágenes y labels por split (train/val/test) y compara contra los números oficiales del plan (~14.100 / 3.100 / 4.300).
4. Verifica que cada imagen tenga su `.txt` de labels correspondiente (y viceversa).
5. Lee labels reales para confirmar el orden de los índices de clase (`smoke`=0, `fire`=1 o al revés).

## 1. Descarga

Usamos `kagglehub`, que maneja la autenticación con Kaggle (pide usuario y API key la primera vez: Kaggle > Settings > Create New Token, descarga `kaggle.json`). `kagglehub` primero baja el dataset a su cache local; después copiamos ese contenido a `data/`, que es donde el resto del repo va a esperar encontrarlo.

In [ ]:
import os
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import kagglehub
import pandas as pd

DATASET_SLUG = "sayedgamal99/smoke-fire-detection-yolo"

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR

os.environ["KAGGLEHUB_CACHE"] = str(DATA_DIR)

DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_SLUG))
print("Dataset descargado en:", DATASET_ROOT)

WindowsPath('c:/Proyectos/CV2/detección_incendios/data')

## 2. Estructura de carpetas

Antes de asumir nada sobre `train/val/test` o `images/labels`, listamos el árbol real tal como quedó en `data/`.

In [4]:
def print_tree(path: Path, prefix: str = "", max_depth: int = 4, depth: int = 0, max_entries: int = 15):
    if depth >= max_depth:
        return
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for i, entry in enumerate(entries[:max_entries]):
        connector = "└── " if i == len(entries) - 1 else "├── "
        suffix = ""
        if entry.is_dir():
            n_items = sum(1 for _ in entry.iterdir())
            suffix = f"  ({n_items} items)"
        print(prefix + connector + entry.name + suffix)
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, prefix + extension, max_depth, depth + 1, max_entries)
    if len(entries) > max_entries:
        print(prefix + f"... (+{len(entries) - max_entries} más)")

print_tree(DATASET_ROOT)

├── data  (3 items)
│   ├── test  (2 items)
│   │   ├── images  (4306 items)
│   │   │   ├── AoF06723.jpg
│   │   │   ├── AoF06724.jpg
│   │   │   ├── AoF06725.jpg
│   │   │   ├── AoF06726.jpg
│   │   │   ├── AoF06727.jpg
│   │   │   ├── AoF06728.jpg
│   │   │   ├── AoF06729.jpg
│   │   │   ├── AoF06730.jpg
│   │   │   ├── AoF06731.jpg
│   │   │   ├── AoF06732.jpg
│   │   │   ├── AoF06733.jpg
│   │   │   ├── AoF06734.jpg
│   │   │   ├── AoF06735.jpg
│   │   │   ├── AoF06736.jpg
│   │   │   ├── AoF06737.jpg
│   │   │   ... (+4291 más)
│   │   └── labels  (4306 items)
│   │       ├── AoF06723.txt
│   │       ├── AoF06724.txt
│   │       ├── AoF06725.txt
│   │       ├── AoF06726.txt
│   │       ├── AoF06727.txt
│   │       ├── AoF06728.txt
│   │       ├── AoF06729.txt
│   │       ├── AoF06730.txt
│   │       ├── AoF06731.txt
│   │       ├── AoF06732.txt
│   │       ├── AoF06733.txt
│   │       ├── AoF06734.txt
│   │       ├── AoF06735.txt
│   │       ├── AoF06736.txt
│   │       ├── AoF06

## 3. Ubicar splits e imágenes/labels

A partir del árbol impreso arriba, ajustar `SPLIT_DIRS` si los nombres de carpeta no coinciden con lo esperado (`train`, `val`, `test`) o si `images`/`labels` están anidados distinto.

In [5]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def find_split_dirs(root: Path):
    """Busca subcarpetas cuyo nombre matchee train/val/test en todo el árbol."""
    candidates = {"train": [], "val": [], "valid": [], "test": []}
    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() in candidates:
            candidates[p.name.lower()].append(p)
    return candidates

split_candidates = find_split_dirs(DATASET_ROOT)
for name, dirs in split_candidates.items():
    for d in dirs:
        print(f"{name}: {d.relative_to(DATASET_ROOT)}")

train: data\train
val: data\val
test: data\test


In [8]:
SPLIT_DIRS = {
    "train": split_candidates["train"][0] if split_candidates["train"] else None,
    "val": (split_candidates["val"] or split_candidates["valid"] or [None])[0],
    "test": split_candidates["test"][0] if split_candidates["test"] else None,
}
SPLIT_DIRS

{'train': WindowsPath('c:/Proyectos/CV2/detección_incendios/data/data/train'),
 'val': WindowsPath('c:/Proyectos/CV2/detección_incendios/data/data/val'),
 'test': WindowsPath('c:/Proyectos/CV2/detección_incendios/data/data/test')}

## 4. Conteo de imágenes y labels por split

Referencia del plan: **~14.100 train / 3.100 val / 4.300 test**, total 21.527 imágenes. Si los números no cierran en ese orden, hay que revisar `SPLIT_DIRS` antes de seguir.

In [9]:
def images_and_labels(split_dir: Path):
    images = sorted([p for p in split_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS])
    labels = sorted([p for p in split_dir.rglob("*.txt")])
    return images, labels

rows = []
split_files = {}
for split, d in SPLIT_DIRS.items():
    if d is None:
        print(f"[!] No se encontró carpeta para split '{split}'")
        continue
    images, labels = images_and_labels(d)
    split_files[split] = {"images": images, "labels": labels}
    rows.append({"split": split, "n_imagenes": len(images), "n_labels": len(labels)})

df_counts = pd.DataFrame(rows)
df_counts.loc["total"] = ["total", df_counts["n_imagenes"].sum(), df_counts["n_labels"].sum()]
df_counts

,split,n_imagenes,n_labels
0,train,14122,14122
1,val,3099,3099
2,test,4306,4306
total,total,21527,21527


## 5. Chequeo de correspondencia imagen ↔ label

Cada imagen debe tener un `.txt` con el mismo stem (aunque esté vacío, para las imágenes "none"). Si el `.txt` no existe, D-Fire lo trata como imagen sin objetos igual — pero conviene saber cuántas caen en cada caso antes del EDA.

In [10]:
match_rows = []
for split, files in split_files.items():
    img_stems = {p.stem for p in files["images"]}
    lbl_stems = {p.stem for p in files["labels"]}
    empty_labels = sum(1 for p in files["labels"] if p.stat().st_size == 0)
    match_rows.append({
        "split": split,
        "imagenes_sin_label": len(img_stems - lbl_stems),
        "labels_sin_imagen": len(lbl_stems - img_stems),
        "labels_vacios_none": empty_labels,
    })

pd.DataFrame(match_rows)

,split,imagenes_sin_label,labels_sin_imagen,labels_vacios_none
0,train,0,0,6458
1,val,0,0,1375
2,test,0,0,2005


## 6. Orden real de índices de clase

**Regla dura del plan:** no asumir `names: [smoke, fire]`. Hay que leerlo de los labels reales (y del `data.yaml`/`classes.txt` si el paquete de Kaggle trae uno) antes de escribir `configs/data.yaml`.

In [11]:
# Buscar cualquier archivo de metadata de clases que traiga el dataset
for pattern in ("*.yaml", "*.yml", "classes.txt", "*.names"):
    for p in DATASET_ROOT.rglob(pattern):
        print("---", p.relative_to(DATASET_ROOT), "---")
        print(p.read_text(encoding="utf-8", errors="replace"))
        print()

--- data.yaml ---
path: /kaggle/working/D Fire Dataset  # dataset root dir
train: data/train/images  # train images (relative to 'path')
val: data/val/images  # val images (relative to 'path')
test: data/test/images  # test images (relative to 'path')

# Classes
names: ['smoke', 'fire']  # Replace with your actual class names

# Counts
nc: 2  # number of classes
train_count: 14122
val_count: 3099
test_count: 4306




In [13]:
# Conteo de índices de clase que aparecen realmente en los labels (train, como referencia)
class_counter = Counter()
train_labels = split_files.get("train", {}).get("labels", [])
for lbl_path in train_labels:
    text = lbl_path.read_text().strip()
    if not text:
        continue
    for line in text.splitlines():
        cls_idx = int(line.split()[0])
        class_counter[cls_idx] += 1

print("Boxes por índice de clase (split train):")
for idx, n in sorted(class_counter.items()):
    print(f"  clase {idx}: {n} boxes")
print()

Boxes por índice de clase (split train):
  clase 0: 7794 boxes
  clase 1: 9638 boxes

